# Pathfinding via Reinforcement and Imitation Multi-Agent Learning (PRIMAL)

While training is taking place, statistics on agent performance are available from Tensorboard. To launch it use:

`tensorboard --logdir train_primal`

In [1]:
import matplotlib
print(matplotlib)
print(getattr(matplotlib, "__file__", None))
print(getattr(matplotlib, "__version__", None))

<module 'matplotlib' from '/root/miniconda3/lib/python3.8/site-packages/matplotlib/__init__.py'>
/root/miniconda3/lib/python3.8/site-packages/matplotlib/__init__.py
3.4.3


In [2]:
#this should be the thing, right?
from __future__ import division

import gym
import numpy as np
import random
import tensorflow as tf
import tensorflow.contrib.layers as layers
# import matplotlib.pyplot as plt
try:
    from od_mstar3 import cpp_mstar
    USE_CPP_MSTAR = False
except ImportError:
    cpp_mstar = None
    USE_CPP_MSTAR = False

from od_mstar3 import od_mstar
from od_mstar3.col_set_addition import OutOfTimeError, NoSolutionError
import threading
import time
import scipy.signal as signal
import os
import GroupLock
import multiprocessing
# %matplotlib inline
import mapf_gym as mapf_gym
import pickle
import imageio
from ACNet import ACNet

from tensorflow.python.client import device_lib
dev_list = device_lib.list_local_devices()
print(dev_list)
# assert len(dev_list) > 1

2026-04-18 14:51:59.951594: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0


The TensorFlow contrib module will not be included in TensorFlow 2.0.
For more information, please see:
  * https://github.com/tensorflow/community/blob/master/rfcs/20180907-contrib-sunset.md
  * https://github.com/tensorflow/addons
  * https://github.com/tensorflow/io (for I/O related ops)
If you depend on functionality not listed there, please file an issue.



2026-04-18 14:52:01.251073: I tensorflow/core/platform/cpu_feature_guard.cc:145] Your CPU supports instructions that this TensorFlow binary was not compiled to use: SSE4.1 SSE4.2 AVX
2026-04-18 14:52:01.283025: I tensorflow/core/platform/profile_utils/cpu_utils.cc:94] CPU Frequency: 2400000000 Hz
2026-04-18 14:52:01.288537: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x55ed5e85a480 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-04-18 14:52:01.288568: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Host, Default Version
2026-04-18 14:52:01.290586: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2026-04-18 14:52:01.392732: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1666] Found device 0 with properties: 
name: NVIDIA GeForce RTX 4080 SUPER major: 8 minor: 9 memoryClockRate(GHz): 2.55
pciBusID: 0000:db:00.0
2026-04-18 14:52:

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 15617645872070163909
, name: "/device:XLA_CPU:0"
device_type: "XLA_CPU"
memory_limit: 17179869184
locality {
}
incarnation: 13026365226344815485
physical_device_desc: "device: XLA_CPU device"
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 31514558464
locality {
  bus_id: 2
  numa_node: 1
  links {
  }
}
incarnation: 18411466827126188786
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 4080 SUPER, pci bus id: 0000:db:00.0, compute capability: 8.9"
, name: "/device:XLA_GPU:0"
device_type: "XLA_GPU"
memory_limit: 17179869184
locality {
}
incarnation: 9785979558849745543
physical_device_desc: "device: XLA_GPU device"
]


2026-04-18 14:52:02.046907: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1206] Device interconnect StreamExecutor with strength 1 edge matrix:
2026-04-18 14:52:02.046935: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1212]      0 
2026-04-18 14:52:02.046939: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1225] 0:   N 
2026-04-18 14:52:02.050581: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1351] Created TensorFlow device (/device:GPU:0 with 30054 MB memory) -> physical GPU (device: 0, name: NVIDIA GeForce RTX 4080 SUPER, pci bus id: 0000:db:00.0, compute capability: 8.9)
2026-04-18 14:52:02.053631: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x55ed84983f20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-18 14:52:02.053644: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4080 SUPER, Compute Capability 8.9


### Helper Functions

In [3]:
def make_gif(images, fname, duration=2, true_image=False,salience=False,salIMGS=None):
    imageio.mimwrite(fname,images,subrectangles=True)
    print("wrote gif")

# Copies one set of variables to another.
# Used to set worker network parameters to those of global network.
def update_target_graph(from_scope,to_scope):
    from_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, from_scope)
    to_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, to_scope)

    op_holder = []
    for from_var,to_var in zip(from_vars,to_vars):
        op_holder.append(to_var.assign(from_var))
    return op_holder

def discount(x, gamma):
    return signal.lfilter([1], [1, -gamma], x[::-1], axis=0)[::-1]

def good_discount(x, gamma):
    return discount(x,gamma)

def expert_find_path(world, start_positions, goals, inflation=2, time_limit=5):
    if USE_CPP_MSTAR and cpp_mstar is not None:
        return cpp_mstar.find_path(world, start_positions, goals, inflation, time_limit)
    return od_mstar.find_path(world, start_positions, goals, inflation, time_limit)

## Worker Agent

In [4]:
class Worker:
    def __init__(self, game, metaAgentID, workerID, a_size, groupLock):
        self.workerID = workerID
        self.env = game
        self.metaAgentID = metaAgentID
        self.name = "worker_"+str(workerID)
        self.agentID = ((workerID-1) % num_workers) + 1 
        self.groupLock = groupLock

        self.nextGIF = episode_count # For GIFs output
        #Create the local copy of the network and the tensorflow op to copy global parameters to local network
        self.local_AC = ACNet(self.name,a_size,trainer,True,GRID_SIZE,GLOBAL_NET_SCOPE)
        self.pull_global = update_target_graph(GLOBAL_NET_SCOPE, self.name)

    def synchronize(self):
        #handy thing for keeping track of which to release and acquire
        if(not hasattr(self,"lock_bool")):
            self.lock_bool=False
        self.groupLock.release(int(self.lock_bool),self.name)
        self.groupLock.acquire(int(not self.lock_bool),self.name)
        self.lock_bool=not self.lock_bool
        
    def train(self, rollout, sess, gamma, bootstrap_value, rnn_state0, imitation=False):
        global episode_count
        if imitation:
            rollout=np.array(rollout)
            #we calculate the loss differently for imitation
            #if imitation=True the rollout is assumed to have different dimensions:
            #[o[0],o[1],optimal_actions]
            feed_dict={global_step:episode_count,
                       self.local_AC.inputs:np.stack(rollout[:,0]),
                       self.local_AC.goal_pos:np.stack(rollout[:,1]),
                       self.local_AC.optimal_actions:np.stack(rollout[:,2]),
                       self.local_AC.state_in[0]:rnn_state0[0],
                       self.local_AC.state_in[1]:rnn_state0[1]
                      }
            _,i_l,_=sess.run([self.local_AC.policy,self.local_AC.imitation_loss,
                              self.local_AC.apply_imitation_grads],
                             feed_dict=feed_dict)
            return i_l
        rollout = np.array(rollout)
        observations = rollout[:,0]
        goals=rollout[:,-2]
        actions = rollout[:,1]
        rewards = rollout[:,2]
        values = rollout[:,5]
        valids = rollout[:,6]
        blockings = rollout[:,10]
        # on_goals=rollout[:,8]
        train_value = rollout[:,-1]

        # Here we take the rewards and values from the rollout, and use them to 
        # generate the advantage and discounted returns. (With bootstrapping)
        # The advantage function uses "Generalized Advantage Estimation"
        self.rewards_plus = np.asarray(rewards.tolist() + [bootstrap_value])
        discounted_rewards = discount(self.rewards_plus,gamma)[:-1]
        self.value_plus = np.asarray(values.tolist() + [bootstrap_value])
        advantages = rewards + gamma * self.value_plus[1:] - self.value_plus[:-1]
        advantages = good_discount(advantages,gamma)

        num_samples = min(EPISODE_SAMPLES,len(advantages))
        sampleInd = np.sort(np.random.choice(advantages.shape[0], size=(num_samples,), replace=False))

        # Update the global network using gradients from loss
        # Generate network statistics to periodically save
        feed_dict = {
            global_step:episode_count,
            self.local_AC.target_v:np.stack(discounted_rewards),
            self.local_AC.inputs:np.stack(observations),
            self.local_AC.goal_pos:np.stack(goals),
            self.local_AC.actions:actions,
            self.local_AC.train_valid:np.stack(valids),
            self.local_AC.advantages:advantages,
            self.local_AC.train_value:train_value,
            self.local_AC.target_blockings:blockings,
            # self.local_AC.target_on_goals:on_goals,
            self.local_AC.state_in[0]:rnn_state0[0],
            self.local_AC.state_in[1]:rnn_state0[1]
        }
        
        v_l,p_l,valid_l,e_l,g_n,v_n,b_l,_ = sess.run([self.local_AC.value_loss,
            self.local_AC.policy_loss,
            self.local_AC.valid_loss,
            self.local_AC.entropy,
            self.local_AC.grad_norms,
            self.local_AC.var_norms,
            self.local_AC.blocking_loss,
            self.local_AC.apply_grads],
            feed_dict=feed_dict)
        return v_l/len(rollout), p_l/len(rollout), valid_l/len(rollout), e_l/len(rollout), b_l/len(rollout), g_n, v_n

    def shouldRun(self, coord, episode_count):
        global run_until_episode
        if TRAINING:
            target_episode = MAX_EPISODES if run_until_episode is None else run_until_episode
            return (not coord.should_stop()) and (episode_count < target_episode)
        else:
            return (episode_count < NUM_EXPS)

    def _checkpoint_path(self, episode_idx):
        return model_path + '/model-' + str(int(episode_idx)) + '.cptk'

    def _register_episode_end(self, mode):
        global episode_count, rl_episode_count, il_episode_count
        assert mode in ("RL", "IL")
        with counter_lock:
            episode_count += 1
            current_episode = int(episode_count)
            if mode == "RL":
                rl_episode_count += 1
            else:
                il_episode_count += 1
            rl_count = int(rl_episode_count)
            il_count = int(il_episode_count)
        return current_episode, rl_count, il_count

    def _maybe_save_model(self, sess, saver, current_episode):
        global next_save_episode
        if (not TRAINING) or (self.workerID != 1) or (not AUTO_SAVE_DURING_TRAINING):
            return False
        should_save = False
        with counter_lock:
            if current_episode >= next_save_episode:
                should_save = True
                while current_episode >= next_save_episode:
                    next_save_episode += SAVE_MODEL_EVERY
        if should_save:
            print('Saving Model @ episode {}'.format(current_episode), flush=True)
            saver.save(sess, self._checkpoint_path(current_episode), write_meta_graph=False)
            print('Saved Model', flush=True)
        return should_save

    def _should_write_summary(self, current_episode):
        global next_summary_episode
        if (not TRAINING) or (self.workerID != 1):
            return False
        write_summary = False
        with counter_lock:
            if current_episode >= next_summary_episode:
                write_summary = True
                while current_episode >= next_summary_episode:
                    next_summary_episode += SUMMARY_WINDOW
        return write_summary

    def parse_path(self,path):
        '''needed function to take the path generated from M* and create the 
        observations and actions for the agent
        path: the exact path ouput by M*, assuming the correct number of agents
        returns: the list of rollouts for the "episode": 
                list of length num_agents with each sublist a list of tuples 
                (observation[0],observation[1],optimal_action,reward)'''
        result=[[] for i in range(num_workers)]
        for t in range(len(path[:-1])):
            observations=[]
            move_queue=list(range(num_workers))
            for agent in range(1,num_workers+1):
                observations.append(self.env._observe(agent))
            steps=0
            while len(move_queue)>0:
                steps+=1
                i=move_queue.pop(0)
                o=observations[i]
                pos=path[t][i]
                newPos=path[t+1][i]#guaranteed to be in bounds by loop guard
                direction=(newPos[0]-pos[0],newPos[1]-pos[1])
                a=self.env.world.getAction(direction)
                state, reward, done, nextActions, on_goal, blocking, valid_action=self.env._step((i+1,a))
                if steps>num_workers**2:
                    #if we have a very confusing situation where lots of agents move
                    #in a circle (difficult to parse and also (mostly) impossible to learn)
                    return None
                if not valid_action:
                    #the tie must be broken here
                    move_queue.append(i)
                    continue
                result[i].append([o[0],o[1],a])
        return result
        
    def work(self,max_episode_length,gamma,sess,coord,saver):
        global episode_count, rl_episode_count, il_episode_count
        global next_save_episode, next_summary_episode
        global swarm_reward, episode_rewards, episode_lengths, episode_mean_values
        global episode_invalid_ops, episode_wrong_blocking, episode_done_flags, episode_hit_max_flags
        total_steps, i_buf = 0, 0
        episode_buffers, s1Values = [ [] for _ in range(NUM_BUFFERS) ], [ [] for _ in range(NUM_BUFFERS) ]

        with sess.as_default(), sess.graph.as_default():
            while self.shouldRun(coord, episode_count):
                sess.run(self.pull_global)

                episode_buffer, episode_values = [], []
                episode_reward = episode_step_count = episode_inv_count = 0
                v_l = p_l = valid_l = e_l = b_l = g_n = v_n = np.nan
                i_l = np.nan
                d = False

                # Initial state from the environment
                if self.agentID==1:
                    self.env._reset(self.agentID)
                self.synchronize() # synchronize starting time of the threads
                validActions          = self.env._listNextValidActions(self.agentID)
                s                     = self.env._observe(self.agentID)
                blocking              = False
                p=self.env.world.getPos(self.agentID)
                on_goal               = self.env.world.goals[p[0],p[1]]==self.agentID
                s                     = self.env._observe(self.agentID)
                rnn_state             = self.local_AC.state_init
                rnn_state0            = rnn_state
                RewardNb = 0 
                wrong_blocking  = 0
                # wrong_on_goal=0

                if self.agentID==1:
                    global demon_probs
                    demon_probs[self.metaAgentID]=np.random.rand()
                self.synchronize() # synchronize starting time of the threads

                # reset swarm_reward (for tensorboard)
                swarm_reward[self.metaAgentID] = 0
                demo_episode = (episode_count < PRIMING_LENGTH) or (demon_probs[self.metaAgentID] < DEMONSTRATION_PROB)
                if demo_episode:
                    # For the first PRIMING_LENGTH episodes, or with a certain probability,
                    # observe a demonstration from the expert planner.
                    global rollouts, demo_fail_reasons
                    rollouts[self.metaAgentID] = None
                    demo_fail_reasons[self.metaAgentID] = None
                    if(self.agentID==1):
                        world=self.env.getObstacleMap()
                        start_positions=tuple(self.env.getPositions())
                        goals=tuple(self.env.getGoals())
                        try:
                            mstar_path = expert_find_path(
                                world,
                                start_positions,
                                goals,
                                inflation=2,
                                time_limit=IL_EXPERT_TIME_LIMIT
                            )
                            parsed_rollouts = self.parse_path(mstar_path)
                            if parsed_rollouts is None:
                                demo_fail_reasons[self.metaAgentID] = "parse_path returned None"
                            else:
                                parsed_rollouts = [agent_rollout[:IL_MAX_EPISODE_LENGTH] for agent_rollout in parsed_rollouts]
                                if not all(len(agent_rollout) > 0 for agent_rollout in parsed_rollouts):
                                    demo_fail_reasons[self.metaAgentID] = "parsed demonstration empty for at least one agent"
                                else:
                                    rollouts[self.metaAgentID] = parsed_rollouts
                        except OutOfTimeError as exc:
                            demo_fail_reasons[self.metaAgentID] = "OutOfTimeError: {}".format(repr(exc))
                        except NoSolutionError as exc:
                            demo_fail_reasons[self.metaAgentID] = "NoSolutionError: {}".format(repr(exc))
                        except Exception as exc:
                            demo_fail_reasons[self.metaAgentID] = "{}: {}".format(type(exc).__name__, repr(exc))
                    self.synchronize()
                    if rollouts[self.metaAgentID] is not None:
                        i_l=self.train(rollouts[self.metaAgentID][self.agentID-1], sess, gamma, None, rnn_state0, imitation=True)
                        self.synchronize()
                        if self.workerID == 1:
                            current_episode, rl_count, il_count = self._register_episode_end("IL")
                            self._maybe_save_model(sess, saver, current_episode)

                            summary = tf.Summary()
                            summary.value.add(tag='Losses/Imitation loss', simple_value=i_l)
                            summary.value.add(tag='Perf/Total Episode Count', simple_value=current_episode)
                            summary.value.add(tag='Perf/RL Episode Count', simple_value=rl_count)
                            summary.value.add(tag='Perf/IL Episode Count', simple_value=il_count)
                            summary.value.add(tag='Perf/IL Share', simple_value=(float(il_count) / float(max(1, current_episode))))
                            global_summary.add_summary(summary, current_episode)
                            global_summary.flush()

                            if PRINT_EVERY_EPISODE and (current_episode % EPISODE_LOG_INTERVAL == 0):
                                current_learning_rate = sess.run(lr, feed_dict={global_step: float(current_episode)})
                                print(
                                    "[Episode {:d}] mode=IL | lr={:.6e} | rl_count={} | il_count={} | il_share={:.3f} | loss={:.6f} | demo_len_cap={}".format(
                                        current_episode,
                                        current_learning_rate,
                                        rl_count,
                                        il_count,
                                        float(il_count) / float(max(1, current_episode)),
                                        float(i_l),
                                        int(IL_MAX_EPISODE_LENGTH)
                                    ),
                                    flush=True
                                )
                        self.synchronize()
                        continue

                    self.synchronize()
                    if self.workerID == 1 and PRINT_EVERY_EPISODE:
                        next_episode = int(episode_count) + 1
                        print(
                            "[Episode {:d}] demonstration unavailable -> skip episode | reason={}".format(
                                next_episode,
                                demo_fail_reasons[self.metaAgentID]
                            ),
                            flush=True
                        )
                    self.synchronize()
                    continue
                saveGIF = False
                if OUTPUT_GIFS and self.workerID == 1 and ((not TRAINING) or (episode_count >= self.nextGIF)):
                    saveGIF = True
                    self.nextGIF =episode_count + 64
                    GIF_episode = int(episode_count)
                    episode_frames = [ self.env._render(mode='rgb_array',screen_height=900,screen_width=900) ]
                    
                while (not self.env.finished): # Give me something!
                    #Take an action using probabilities from policy network output.
                    a_dist,v,rnn_state,pred_blocking = sess.run([self.local_AC.policy,
                                                   self.local_AC.value,
                                                   self.local_AC.state_out,
                                                   self.local_AC.blocking], 
                                         feed_dict={self.local_AC.inputs:[s[0]],
                                                    self.local_AC.goal_pos:[s[1]],
                                                    self.local_AC.state_in[0]:rnn_state[0],
                                                    self.local_AC.state_in[1]:rnn_state[1]})

                    if(not (np.argmax(a_dist.flatten()) in validActions)):
                        episode_inv_count += 1
                    train_valid = np.zeros(a_size)
                    train_valid[validActions] = 1

                    valid_dist = np.array([a_dist[0,validActions]])
                    valid_dist /= np.sum(valid_dist)

                    if TRAINING:
                        if (pred_blocking.flatten()[0] < 0.5) == blocking:
                            wrong_blocking += 1
                        # if (pred_on_goal.flatten()[0] < 0.5) == on_goal:
                        #     wrong_on_goal += 1
                        a           = validActions[ np.random.choice(range(valid_dist.shape[1]),p=valid_dist.ravel()) ]
                        train_val   = 1.
                    else:
                        a         = np.argmax(a_dist.flatten())
                        if a not in validActions or not GREEDY:
                            a     = validActions[ np.random.choice(range(valid_dist.shape[1]),p=valid_dist.ravel()) ]
                        train_val = 1.

                    _, r, _, _, on_goal,blocking,_ = self.env._step((self.agentID, a),episode=episode_count)

                    self.synchronize() # synchronize threads

                    # Get common observation for all agents after all individual actions have been performed
                    s1           = self.env._observe(self.agentID)
                    validActions = self.env._listNextValidActions(self.agentID, a,episode=episode_count)
                    d            = self.env.finished

                    if saveGIF:
                        episode_frames.append(self.env._render(mode='rgb_array',screen_width=900,screen_height=900))

                    episode_buffer.append([s[0],a,r,s1,d,v[0,0],train_valid,pred_blocking,int(blocking),s[1],train_val])
                    episode_values.append(v[0,0])
                    episode_reward += r
                    s = s1
                    total_steps += 1
                    episode_step_count += 1

                    if r>0:
                        RewardNb += 1
                    if d == True:
                        print('\n{} Goodbye World. We did it!'.format(episode_step_count), end='\n')

                    # If the episode hasn't ended, but the experience buffer is full, then we
                    # make an update step using that experience rollout.
                    if TRAINING and (len(episode_buffer) % EXPERIENCE_BUFFER_SIZE == 0 or d):
                        # Since we don't know what the true final return is, we "bootstrap" from our current value estimation.
                        if len(episode_buffer) >= EXPERIENCE_BUFFER_SIZE:
                            episode_buffers[i_buf] = episode_buffer[-EXPERIENCE_BUFFER_SIZE:]
                        else:
                            episode_buffers[i_buf] = episode_buffer[:]

                        if d:
                            s1Values[i_buf] = 0
                        else:
                            s1Values[i_buf] = sess.run(self.local_AC.value, 
                                 feed_dict={self.local_AC.inputs:np.array([s[0]])
                                            ,self.local_AC.goal_pos:[s[1]]
                                            ,self.local_AC.state_in[0]:rnn_state[0]
                                            ,self.local_AC.state_in[1]:rnn_state[1]})[0,0]

                        if (episode_count-EPISODE_START) < NUM_BUFFERS:
                            i_rand = np.random.randint(i_buf+1)
                        else:
                            i_rand = np.random.randint(NUM_BUFFERS)
                            tmp = np.array(episode_buffers[i_rand])
                            while tmp.shape[0] == 0:
                                i_rand = np.random.randint(NUM_BUFFERS)
                                tmp = np.array(episode_buffers[i_rand])
                        v_l,p_l,valid_l,e_l,b_l,g_n,v_n = self.train(episode_buffers[i_rand],sess,gamma,s1Values[i_rand],rnn_state0)

                        i_buf = (i_buf + 1) % NUM_BUFFERS
                        rnn_state0             = rnn_state
                        episode_buffers[i_buf] = []

                    self.synchronize() # synchronize threads
                    # sess.run(self.pull_global)
                    if episode_step_count >= max_episode_length or d:
                        break

                episode_lengths[self.metaAgentID].append(episode_step_count)
                episode_mean_values[self.metaAgentID].append(np.nanmean(episode_values) if len(episode_values) > 0 else np.nan)
                episode_invalid_ops[self.metaAgentID].append(episode_inv_count)
                episode_wrong_blocking[self.metaAgentID].append(wrong_blocking)
                episode_done = bool(d)
                episode_hit_max = bool((episode_step_count >= max_episode_length) and (not d))
                episode_done_flags[self.metaAgentID].append(int(episode_done))
                episode_hit_max_flags[self.metaAgentID].append(int(episode_hit_max))

                # Periodically save gifs of episodes, model parameters, and summary statistics.
                if episode_count % EXPERIENCE_BUFFER_SIZE == 0 and printQ:
                    print('                                                                                   ', end='\r')
                    print('{} Episode terminated ({},{})'.format(episode_count, self.agentID, RewardNb), end='\r')

                swarm_reward[self.metaAgentID] += episode_reward

                self.synchronize() # synchronize threads

                episode_rewards[self.metaAgentID].append(swarm_reward[self.metaAgentID])

                if not TRAINING:
                    mutex.acquire()
                    if episode_count < NUM_EXPS:
                        plan_durations[episode_count] = episode_step_count
                    if self.workerID == 1:
                        episode_count += 1
                        print('({}) Thread {}: {} steps, {:.2f} reward ({} invalids).'.format(episode_count, self.workerID, episode_step_count, episode_reward, episode_inv_count))
                    GIF_episode = int(episode_count)
                    mutex.release()
                else:
                    self.synchronize()
                    if self.workerID == 1:
                        current_episode, rl_count, il_count = self._register_episode_end("RL")
                        self._maybe_save_model(sess, saver, current_episode)

                        if PRINT_EVERY_EPISODE and (current_episode % EPISODE_LOG_INTERVAL == 0):
                            current_learning_rate = sess.run(lr, feed_dict={global_step: float(current_episode)})
                            total_loss = float(v_l + p_l + valid_l + b_l)
                            print(
                                "[Episode {:d}] mode=RL | lr={:.6e} | reward={:.2f} | loss={:.6f} | length={} | done={} | hit_max={} | rl_count={} | il_count={} | il_share={:.3f}".format(
                                    current_episode,
                                    current_learning_rate,
                                    episode_reward,
                                    total_loss,
                                    episode_step_count,
                                    episode_done,
                                    episode_hit_max,
                                    rl_count,
                                    il_count,
                                    float(il_count) / float(max(1, current_episode))
                                ),
                                flush=True
                            )

                        if self._should_write_summary(current_episode):
                            SL = SUMMARY_WINDOW * num_workers
                            mean_reward = np.nanmean(episode_rewards[self.metaAgentID][-SL:])
                            mean_length = np.nanmean(episode_lengths[self.metaAgentID][-SL:])
                            mean_value = np.nanmean(episode_mean_values[self.metaAgentID][-SL:])
                            mean_invalid = np.nanmean(episode_invalid_ops[self.metaAgentID][-SL:])
                            mean_wrong_blocking = np.nanmean(episode_wrong_blocking[self.metaAgentID][-SL:])
                            done_rate = np.nanmean(episode_done_flags[self.metaAgentID][-SL:])
                            hit_max_rate = np.nanmean(episode_hit_max_flags[self.metaAgentID][-SL:])
                            current_learning_rate = sess.run(lr, feed_dict={global_step: float(current_episode)})

                            summary = tf.Summary()
                            summary.value.add(tag='Perf/Learning Rate', simple_value=current_learning_rate)
                            summary.value.add(tag='Perf/Reward', simple_value=mean_reward)
                            summary.value.add(tag='Perf/Length', simple_value=mean_length)
                            summary.value.add(tag='Perf/Done Rate', simple_value=done_rate)
                            summary.value.add(tag='Perf/Hit Max Step Rate', simple_value=hit_max_rate)
                            summary.value.add(tag='Perf/RL Episode Count', simple_value=rl_count)
                            summary.value.add(tag='Perf/IL Episode Count', simple_value=il_count)
                            summary.value.add(tag='Perf/IL Share', simple_value=(float(il_count) / float(max(1, current_episode))))
                            summary.value.add(tag='Perf/Valid Rate', simple_value=(mean_length-mean_invalid)/mean_length)
                            summary.value.add(tag='Perf/Blocking Prediction Accuracy', simple_value=(mean_length-mean_wrong_blocking)/mean_length)

                            summary.value.add(tag='Losses/Value Loss', simple_value=v_l)
                            summary.value.add(tag='Losses/Policy Loss', simple_value=p_l)
                            summary.value.add(tag='Losses/Blocking Loss', simple_value=b_l)
                            # summary.value.add(tag='Losses/On Goal Loss', simple_value=og_l)
                            summary.value.add(tag='Losses/Valid Loss', simple_value=valid_l)
                            summary.value.add(tag='Losses/Grad Norm', simple_value=g_n)
                            summary.value.add(tag='Losses/Var Norm', simple_value=v_n)
                            global_summary.add_summary(summary, current_episode)

                            global_summary.flush()

                            if printQ:
                                print('{} Tensorboard updated ({})'.format(current_episode, self.workerID), end='\r')
                    self.synchronize()

                if saveGIF:
                    # Dump episode frames for external gif generation (otherwise, makes the jupyter kernel crash)
                    time_per_step = 0.1
                    images = np.array(episode_frames)
                    if TRAINING:
                        make_gif(images, '{}/episode_{:d}_{:d}_{:.1f}.gif'.format(gifs_path,GIF_episode,episode_step_count,swarm_reward[self.metaAgentID]))
                    else:
                        make_gif(images, '{}/episode_{:d}_{:d}.gif'.format(gifs_path,GIF_episode,episode_step_count), duration=len(images)*time_per_step,true_image=True,salience=False)
                if SAVE_EPISODE_BUFFER:
                    with open('gifs3D/episode_{}.dat'.format(GIF_episode), 'wb') as file:
                        pickle.dump(episode_buffer, file)


## Training

In [5]:
# Learning parameters
max_episode_length     = 256
episode_count          = 0  # total completed episodes (integer, incremented once per episode by worker 1)
rl_episode_count       = 0  # completed RL episodes in current session
il_episode_count       = 0  # completed IL episodes in current session
EPISODE_START          = episode_count
gamma                  = .95 # discount rate for advantage estimation and reward discounting
#moved network parameters to ACNet.py
EXPERIENCE_BUFFER_SIZE = 128
GRID_SIZE              = 10 #the size of the FOV grid to apply to each agent
ENVIRONMENT_SIZE       = (10,40)#the total size of the environment (length of one side)
OBSTACLE_DENSITY       = (0,.3) #range of densities
DIAG_MVMT              = False # Diagonal movements allowed?
a_size                 = 5 + int(DIAG_MVMT)*4
SUMMARY_WINDOW         = 5
SAVE_MODEL_EVERY       = 100  # kept for reference; auto-save can be disabled below

AUTO_SAVE_DURING_TRAINING = False  # avoid save-time OOM while workers/GPU are active
PRINT_EVERY_EPISODE    = True
EPISODE_LOG_INTERVAL   = 1
NUM_META_AGENTS        = 1
NUM_THREADS            = 2 #int(multiprocessing.cpu_count() / (2 * NUM_META_AGENTS))
NUM_BUFFERS            = 1 # NO EXPERIENCE REPLAY int(NUM_THREADS / 2)
EPISODE_SAMPLES        = EXPERIENCE_BUFFER_SIZE # 64
LR_Q                   = 2.e-5 #8.e-5 / NUM_THREADS # default: 1e-5
ADAPT_LR               = True
ADAPT_COEFF            = 5.e-5 #the coefficient A in LR_Q/sqrt(A*steps+1) for calculating LR
load_model             = False
RESET_TRAINER          = False

RUN_NAME   = "2026-04-18_obsfull_rlil_seg01"
BASE_DIR   = "/root/autodl-tmp/experiments/" + RUN_NAME

model_path = BASE_DIR + "/checkpoints"
train_path = BASE_DIR + "/tb"
gifs_path  = BASE_DIR + "/plots"
GLOBAL_NET_SCOPE       = 'global'

#Imitation options
PRIMING_LENGTH         = 2    # number of episodes at the beginning to train only on demonstrations
DEMONSTRATION_PROB     = 0.1  # probability of training on a demonstration per episode
IL_MAX_EPISODE_LENGTH = 64   # truncate expert demonstrations to a shorter horizon
IL_EXPERT_TIME_LIMIT   = 5    # seconds for expert planner when generating demonstrations

# Simulation options
FULL_HELP              = False
OUTPUT_GIFS            = False
SAVE_EPISODE_BUFFER    = False

# Testing
TRAINING               = True
GREEDY                 = False
NUM_EXPS               = 100
MODEL_NUMBER           = 100
MAX_EPISODES           = 1000
SEGMENT_EPISODES       = 100  # run one segment at a time; training cell will stop at episode_count + SEGMENT_EPISODES
DEVICE                 = "/gpu:0"

# Shared counters / thresholds (worker 1 updates these under counter_lock)
run_until_episode      = None  # set at segment start to episode_count + SEGMENT_EPISODES
next_save_episode      = SAVE_MODEL_EVERY
next_summary_episode   = SUMMARY_WINDOW
counter_lock           = threading.Lock()

# Shared arrays for tensorboard
episode_rewards        = [ [] for _ in range(NUM_META_AGENTS) ]
episode_lengths        = [ [] for _ in range(NUM_META_AGENTS) ]
episode_mean_values    = [ [] for _ in range(NUM_META_AGENTS) ]
episode_invalid_ops    = [ [] for _ in range(NUM_META_AGENTS) ]
episode_wrong_blocking = [ [] for _ in range(NUM_META_AGENTS) ]
episode_done_flags     = [ [] for _ in range(NUM_META_AGENTS) ]
episode_hit_max_flags  = [ [] for _ in range(NUM_META_AGENTS) ]
rollouts               = [ None for _ in range(NUM_META_AGENTS)]
demo_fail_reasons      = [ None for _ in range(NUM_META_AGENTS)]
demon_probs=[np.random.rand() for _ in range(NUM_META_AGENTS)]
# episode_steps_on_goal  = [ [] for _ in range(NUM_META_AGENTS) ]
printQ                 = False # (for headless)
swarm_reward           = [0]*NUM_META_AGENTS

In [6]:
tf.reset_default_graph()
print("Hello World")

# If this setup cell is re-run, close the previous live session first.
if 'sess' in globals() and sess is not None:
    try:
        sess.close()
        print("Closed previous session.")
    except Exception as exc:
        print("Previous session close skipped:", repr(exc))

sess = None
coord = None
worker_threads = []

if not os.path.exists(model_path):
    os.makedirs(model_path)
config = tf.ConfigProto(allow_soft_placement = True)
config.gpu_options.allow_growth = True

if not TRAINING:
    plan_durations = np.array([0 for _ in range(NUM_EXPS)])
    mutex = threading.Lock()
    gifs_path += '_tests'
    if SAVE_EPISODE_BUFFER and not os.path.exists('gifs3D'):
        os.makedirs('gifs3D')

# Create a directory to save episode playback gifs to
if not os.path.exists(gifs_path):
    os.makedirs(gifs_path)

with tf.device(DEVICE):
    master_network = ACNet(GLOBAL_NET_SCOPE, a_size, None, False, GRID_SIZE, GLOBAL_NET_SCOPE)

    global_step = tf.placeholder(tf.float32)
    if ADAPT_LR:
        lr = tf.divide(tf.constant(LR_Q), tf.sqrt(tf.add(1., tf.multiply(tf.constant(ADAPT_COEFF), global_step))))
    else:
        lr = tf.constant(LR_Q)
    trainer = tf.contrib.opt.NadamOptimizer(learning_rate=lr, use_locking=True)

    if TRAINING:
        num_workers = NUM_THREADS
    else:
        num_workers = NUM_THREADS
        NUM_META_AGENTS = 1

    gameEnvs, workers, groupLocks = [], [], []
    n = 1
    for ma in range(NUM_META_AGENTS):
        num_agents = NUM_THREADS
        gameEnv = mapf_gym.MAPFEnv(
            num_agents=num_agents,
            DIAGONAL_MOVEMENT=DIAG_MVMT,
            SIZE=ENVIRONMENT_SIZE,
            observation_size=GRID_SIZE,
            PROB=OBSTACLE_DENSITY,
            FULL_HELP=FULL_HELP
        )
        gameEnvs.append(gameEnv)

        workerNames = ["worker_" + str(i) for i in range(n, n + num_workers)]
        groupLock = GroupLock.GroupLock([workerNames, workerNames])
        groupLocks.append(groupLock)

        workersTmp = []
        for i in range(ma * num_workers + 1, (ma + 1) * num_workers + 1):
            workersTmp.append(Worker(gameEnv, ma, n, a_size, groupLock))
            n += 1
        workers.append(workersTmp)

    global_summary = tf.summary.FileWriter(train_path)
    save_vars = tf.get_collection(tf.GraphKeys.GLOBAL_VARIABLES, GLOBAL_NET_SCOPE + '/qvalues')
    print('num save vars =', len(save_vars))
    saver = tf.train.Saver(var_list=save_vars, max_to_keep=5)

sess = tf.Session(config=config)
sess.run(tf.global_variables_initializer())

if load_model == True:
    print('Loading Model...')
    if not TRAINING:
        with open(model_path + '/checkpoint', 'w') as file:
            file.write('model_checkpoint_path: "model-{}.cptk"'.format(MODEL_NUMBER))
    ckpt = tf.train.get_checkpoint_state(model_path)
    p = ckpt.model_checkpoint_path
    p = p[p.find('-') + 1:]
    p = p[:p.find('.')]
    episode_count = int(p)
    rl_episode_count = 0
    il_episode_count = 0
    next_save_episode = ((episode_count // SAVE_MODEL_EVERY) + 1) * SAVE_MODEL_EVERY
    next_summary_episode = ((episode_count // SUMMARY_WINDOW) + 1) * SUMMARY_WINDOW
    saver.restore(sess, ckpt.model_checkpoint_path)
    print("episode_count set to ", episode_count)
    print("rl_episode_count / il_episode_count reset to 0 for this resumed session")
    if RESET_TRAINER:
        trainer = tf.contrib.opt.NadamOptimizer(learning_rate=lr, use_locking=True)

def run_training_segment(segment_episodes=SEGMENT_EPISODES):
    global coord, worker_threads, run_until_episode
    if sess is None:
        raise RuntimeError("Session is not initialized. Re-run the setup cell first.")
    if not TRAINING:
        raise RuntimeError("run_training_segment is only intended for TRAINING=True.")
    segment_episodes = int(segment_episodes)
    run_until_episode = int(episode_count) + segment_episodes
    print("Starting training segment: current episode = {}, run_until_episode = {}".format(int(episode_count), run_until_episode))
    if not AUTO_SAVE_DURING_TRAINING:
        print("AUTO_SAVE_DURING_TRAINING = False -> no checkpoint will be written during the live segment.", flush=True)

    coord = tf.train.Coordinator()
    worker_threads = []
    for ma in range(NUM_META_AGENTS):
        for worker in workers[ma]:
            groupLocks[ma].acquire(0, worker.name)
            print("Starting worker " + str(worker.workerID))
            t = threading.Thread(
                target=worker.work,
                args=(max_episode_length, gamma, sess, coord, saver)
            )
            t.start()
            worker_threads.append(t)
    coord.join(worker_threads)
    print("Training segment finished at episode_count = {}. Session is still alive; run the manual save cell if desired.".format(int(episode_count)))

def manual_save_checkpoint(tag=None, write_meta_graph=None):
    if sess is None:
        raise RuntimeError("Session is not initialized.")
    episode_idx = int(episode_count)
    if tag is None or str(tag).strip() == "":
        save_path = model_path + '/model-' + str(episode_idx) + '.cptk'
    else:
        save_path = model_path + '/model-{}-{}.cptk'.format(str(tag), episode_idx)

    checkpoint_exists = tf.train.get_checkpoint_state(model_path) is not None
    if write_meta_graph is None:
        write_meta_graph = (not checkpoint_exists)

    print("Manual save start ->", save_path, "| write_meta_graph =", write_meta_graph, flush=True)
    saved_path = saver.save(sess, save_path, write_meta_graph=write_meta_graph)
    print("Manual save finished ->", saved_path, flush=True)
    return saved_path

def close_training_session():
    global sess
    if sess is not None:
        sess.close()
        sess = None
        print("Training session closed.")

if not TRAINING:
    print([np.mean(plan_durations), np.sqrt(np.var(plan_durations)), np.mean(np.asarray(plan_durations < max_episode_length, dtype=float))])



Hello World




Hello World... From  global




Hello World... From  worker_1
Hello World... From  worker_2

num save vars = 75





2026-04-18 14:52:09.443499: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1666] Found device 0 with properties: 
name: NVIDIA GeForce RTX 4080 SUPER major: 8 minor: 9 memoryClockRate(GHz): 2.55
pciBusID: 0000:db:00.0
2026-04-18 14:52:09.443581: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2026-04-18 14:52:09.443608: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2026-04-18 14:52:09.443616: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcufft.so.10
2026-04-18 14:52:09.443622: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcurand.so.10
2026-04-18 14:52:09.443629: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcusolver.so.11
2026-04-18 14:52:09.443635: I tensorflow/stream_executo

In [ ]:
# Run one training segment
run_training_segment(SEGMENT_EPISODES)

Starting training segment: current episode = 0, run_until_episode = 100
AUTO_SAVE_DURING_TRAINING = False -> no checkpoint will be written during the live segment.
Starting worker 1
Starting worker 2


2026-04-18 14:52:19.457117: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2026-04-18 14:52:20.405153: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudnn.so.8
2026-04-18 14:52:22.728545: W tensorflow/stream_executor/cuda/redzone_allocator.cc:312] Internal: ptxas exited with non-zero error code 65280, output: ptxas fatal   : Value 'sm_89' is not defined for option 'gpu-name'

Relying on driver to perform ptx compilation. This message will be only logged once.



[Episode 1] mode=IL | lr=1.999950e-05 | rl_count=0 | il_count=1 | il_share=1.000 | loss=1.656250 | demo_len_cap=64
[Episode 2] mode=IL | lr=1.999900e-05 | rl_count=0 | il_count=2 | il_share=1.000 | loss=1.603619 | demo_len_cap=64
[Episode 3] mode=IL | lr=1.999850e-05 | rl_count=0 | il_count=3 | il_share=1.000 | loss=1.597789 | demo_len_cap=64
[Episode 4] mode=RL | lr=1.999800e-05 | reward=-90.30 | loss=118.356923 | length=256 | done=False | hit_max=True | rl_count=1 | il_count=3 | il_share=0.750
[Episode 5] mode=RL | lr=1.999750e-05 | reward=-91.40 | loss=78.105523 | length=256 | done=False | hit_max=True | rl_count=2 | il_count=3 | il_share=0.600
[Episode 6] mode=RL | lr=1.999700e-05 | reward=-89.00 | loss=71.355216 | length=256 | done=False | hit_max=True | rl_count=3 | il_count=3 | il_share=0.500
[Episode 7] mode=RL | lr=1.999650e-05 | reward=-91.80 | loss=41.126825 | length=256 | done=False | hit_max=True | rl_count=4 | il_count=3 | il_share=0.429
[Episode 8] mode=RL | lr=1.999600

## Manual checkpoint save

当一段训练自然结束后，运行下一格手动保存。  
因为此时 workers 已停止、GPU 训练也结束，所以比在训练中途自动保存更稳。

In [1]:
# Manual checkpoint save after the segment finishes
# tag can be changed, e.g. "seg1", "seg2", "final"
manual_save_checkpoint(tag="seg1")

NameError: name 'manual_save_checkpoint' is not defined

## Optional: close the live session

如果本次训练与保存都结束了，可以运行下一格释放 session。

In [ ]:
# Optional cleanup
close_training_session()